In [ ]:
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
warnings.filterwarnings('ignore')

## Set Your Prediction Month Here

In [ ]:
PREDICT_YEAR  = 2026
PREDICT_MONTH = 3   

In [ ]:
# Load saved model and all metadata
MODEL_DIR = './models'

try:
    sales_model   = joblib.load(f'{MODEL_DIR}/sales_xgb.pkl')
    profit_model  = joblib.load(f'{MODEL_DIR}/profit_xgb.pkl')
    FEATURE_COLS  = joblib.load(f'{MODEL_DIR}/feature_cols.pkl')
    PROD_MAP      = joblib.load(f'{MODEL_DIR}/prod_map.pkl')
    SEG_ENC       = joblib.load(f'{MODEL_DIR}/seg_enc.pkl')
    ACTIVE_MONTHS = joblib.load(f'{MODEL_DIR}/active_months.pkl')
    PRODUCT_SPECS = joblib.load(f'{MODEL_DIR}/product_specs.pkl')
    PRODUCT_DISC  = joblib.load(f'{MODEL_DIR}/product_disc.pkl')
    BASE_YEAR     = joblib.load(f'{MODEL_DIR}/base_year.pkl')
    SEASON_NAME   = joblib.load(f'{MODEL_DIR}/season_name.pkl')
    SEASON_ENC    = joblib.load(f'{MODEL_DIR}/season_enc.pkl')
    ALL_PRODUCTS  = joblib.load(f'{MODEL_DIR}/all_products.pkl')
    DISC_ENC      = joblib.load(f'{MODEL_DIR}/disc_enc.pkl')
    DISC_RATE     = joblib.load(f'{MODEL_DIR}/disc_rate.pkl')
    df_raw        = joblib.load(f'{MODEL_DIR}/df_raw.pkl')
    print('Model loaded successfully.')
except FileNotFoundError:
    print('ERROR: Model not found.')
    print('Please run train.ipynb first, then come back here.')
    raise

In [ ]:
# Validate input
assert 1 <= PREDICT_MONTH <= 12, 'Month must be between 1 and 12'
assert PREDICT_YEAR >= BASE_YEAR, f'Year must be {BASE_YEAR} or later'

pred_date   = pd.Timestamp(f'{PREDICT_YEAR}-{PREDICT_MONTH:02d}-01')
season      = SEASON_NAME[PREDICT_MONTH]
active_prods= [p for p in ALL_PRODUCTS if PREDICT_MONTH in ACTIVE_MONTHS[p]]

print(f'Predicting : {pred_date.strftime("%B %Y")}')
print(f'Season     : {season}')
print(f'Active products this month: {len(active_prods)}')
print(f'  {active_prods}')

In [ ]:
def predict_product(year, month, product):
    hist = df_raw[df_raw['Product']==product].copy()
    hist['Date'] = pd.to_datetime(hist['Year'].astype(str)+'-'+hist['Month Number'].astype(str).str.zfill(2)+'-01')
    hist = hist.sort_values('Date').reset_index(drop=True)

    spec   = PRODUCT_SPECS[product]
    mfg    = spec['Manufacturing Price']
    sprice = spec['Sale Price']
    seg    = spec['Segment']
    disc   = PRODUCT_DISC[product]

    same_mo = hist[hist['Month Number']==month]
    units   = same_mo['Units Sold'].mean() if not same_mo.empty else hist['Units Sold'].mean()
    quarter = (month-1)//3+1
    td      = pd.Timestamp(f'{year}-{month:02d}-01')

    def get_lag(col, lag):
        cutoff = td - pd.DateOffset(months=lag)
        row = hist[hist['Date']==cutoff]
        if not row.empty: return row[col].values[0]
        sm = hist[hist['Month Number']==cutoff.month]
        return sm[col].mean() if not sm.empty else hist[col].mean()

    sl = {f'Sales_lag{l}':  get_lag('Sales',l)  for l in [1,2,3,6,12]}
    pl = {f'Profit_lag{l}': get_lag('Profit',l) for l in [1,2,3,6,12]}
    sv, pv = hist['Sales'].values, hist['Profit'].values

    def sm(a,n): return np.mean(a[-n:]) if len(a)>=n else np.mean(a)
    def ss(a,n): return np.std(a[-n:])  if len(a)>=n else 0.0

    rolls = {
        'Sales_roll3_mean':sm(sv,3), 'Sales_roll3_std':ss(sv,3),
        'Sales_roll6_mean':sm(sv,6), 'Sales_roll6_std':ss(sv,6),
        'Profit_roll3_mean':sm(pv,3),'Profit_roll3_std':ss(pv,3),
        'Profit_roll6_mean':sm(pv,6),'Profit_roll6_std':ss(pv,6),
    }

    dr = DISC_RATE.get(disc,0)
    eg = units*sprice

    row = {
        'Units Sold':units, 'Manufacturing Price':mfg, 'Sale Price':sprice,
        'Gross Sales':eg, 'Discounts':eg*dr, 'COGS':units*mfg,
        'Month':month, 'Year_num':year, 'Quarter':quarter,
        'Month_sin':np.sin(2*np.pi*month/12), 'Month_cos':np.cos(2*np.pi*month/12),
        'Qtr_sin':np.sin(2*np.pi*quarter/4),  'Qtr_cos':np.cos(2*np.pi*quarter/4),
        'Season_enc':SEASON_ENC[SEASON_NAME[month]],
        'Year_trend':year-BASE_YEAR,
        'Segment_enc':SEG_ENC.get(seg,0), 'Product_enc':PROD_MAP.get(product,0),
        'DiscBand_enc':DISC_ENC.get(disc,0), 'Discount_Rate':dr,
        'Revenue_per_Unit':sprice*(1-dr), 'Cost_per_Unit':mfg,
        'Price_to_ManufCost':sprice/mfg if mfg else 0,
        'Gross_Margin_pct':(eg-units*mfg)/eg if eg else 0,
        **sl, **pl, **rolls,
    }

    X  = pd.DataFrame([row])[FEATURE_COLS]
    ps = round(float(sales_model.predict(X)[0]),  2)
    pp = round(float(profit_model.predict(X)[0]), 2)

    return {
        'Product':          product,
        'Segment':          seg,
        'Nepal_Season':     SEASON_NAME[month],
        'Discount_Band':    disc,
        'Units_Sold_Est':   round(units, 1),
        'Predicted_Sales':  ps,
        'Predicted_Profit': pp,
        'Margin_pct':       round(pp/ps*100, 1) if ps else 0,
    }

print('Prediction function ready.')

In [ ]:
# Run predictions for all active products in the selected month
results = [predict_product(PREDICT_YEAR, PREDICT_MONTH, p) for p in active_prods]
df_out  = pd.DataFrame(results).sort_values('Predicted_Sales', ascending=False).reset_index(drop=True)

ts = df_out['Predicted_Sales'].sum()
tp = df_out['Predicted_Profit'].sum()

print(f'\n{"="*72}')
print(f'  PREDICTIONS  —  {pred_date.strftime("%B %Y").upper()}')
print(f'  Nepal Season :  {season}')
print(f'{"="*72}')
print(f'  {"Product":<22} {"Segment":<15} {"Sales":>14} {"Profit":>14} {"Margin":>7}')
print(f'  {"-"*72}')
for _, r in df_out.iterrows():
    print(f'  {r["Product"]:<22} {r["Segment"]:<15} '
          f'{r["Predicted_Sales"]:>14,.0f} {r["Predicted_Profit"]:>14,.0f} {r["Margin_pct"]:>6.1f}%')
print(f'  {"-"*72}')
print(f'  {"TOTAL":<37} {ts:>14,.0f} {tp:>14,.0f} {tp/ts*100:>6.1f}%')
print(f'{"="*72}')

In [ ]:
# Display as a clean table
display(df_out)

In [ ]:
# Save results to Excel
import os
os.makedirs('./outputs', exist_ok=True)
out_path = f"./outputs/prediction_{pred_date.strftime('%Y_%m')}.xlsx"
df_out.to_excel(out_path, index=False)
print(f'Saved to: {out_path}')